# Breit-Wigner in Jordi's Code

Let's parametrise the Breit-Wigner using Jordi's formulation in numpy and tensorflow

Jordi Garra Tico's amplitude is defined as:

$$
\begin{align*}
BW(m) &= f_R f_D \frac{1}{(m_0^2 - m^2) + i m_0 \Gamma(m)} \Omega\\
\Gamma(m) &= \Gamma_0 \left(\frac{p}{p_0}\right)^3 \frac{m_0}{m} f_R^2 \\
f_R &= \sqrt{\frac{1+R_r^2p_0^2}{1+R_r^2p^2}} \quad \text{with} \quad R_r = 1.5 \\
f_D &= \sqrt{\frac{1+R_D^2 p_{D,0}^2}{1+R_D^2p_D^2}} \quad \text{with} \quad R_D = 5.0 \\
p &= \sqrt{\frac{(m^2-m_A^2-m_B^2)^2 - 4 m_A^2 m_B^2}{4 m^2}}\\
p_R &= \sqrt{\frac{(m_0^2-m_A^2-m_B^2)^2 - 4m_A^2m_B^2}{4 m_0^2}}\\
p_{D,0} &= \sqrt{\frac{(m_D^2-m_0^2-m_C^2)^2 - 4m_0^2m_C^2}{4m_D^2}}\\
p_D &= \sqrt{\frac{(m_D^2-m^2-m_C^2)^2 - 4m^2m_C^2}{4m_D^2}}\\
\Omega &= m_{AC}^2-m_{BC}^2+\frac{(m_D^2-m_C^2)(m_B^2-m_A^2)}{m^2}
\end{align*}
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

define the phase-space

In [ ]:
# Masses of final state particles
import amplitf.interface as atfi
import amplitf.dynamics as atfd
import amplitf.kinematics as atfk
from amplitf.phasespace.dalitz_phasespace import DalitzPhaseSpace
from particle.particle import literals as lp
# Dalitz Particles
mkz = atfi.const(lp.K_S_0.mass/1000)
mpi = atfi.const(lp.pi_plus.mass/1000)
md = atfi.const(lp.D_0.mass/1000)
# Dalitz Phase Space
phsp = DalitzPhaseSpace(mpi, mkz, mpi, md)

In [ ]:
def density_np(a):
    return np.abs(a)**2

def phase_np(a):
    return np.angle(a)

In [ ]:
pi180inv = np.pi / 180.

def helicity_np(mAB,mBC,mAC,mA,mB,mC,mD,spin):
    hel = np.where( spin==1, mAC*mAC-mBC*mBC+((mD*mD-mC*mC)*(mB*mB-mA*mA)/(mAB*mAB)),
                  np.where( spin==2, np.power(mBC*mBC-mAC*mAC+(mD*mD-mC*mC)*(mA*mA-mB*mB)/(mAB*mAB),2)-
                                    1./3.*(mAB*mAB-2.*(mD*mD+mC*mC)+np.power(mD*mD-mC*mC,2)/(mAB*mAB))*
                                    (mAB*mAB-2.*(mA*mA+mB*mB)+pow(mA*mA-mB*mB,2)/(mAB*mAB)),
                             1.0) )    
    return hel

def two_body_momentum_np(mAB,mA,mB):
    pAB = (((mAB*mAB-mA*mA-mB*mB)*(mAB*mAB-mA*mA-mB*mB)/4.0) - mA*mA*mB*mB)/(mAB*mAB)
    return np.where(pAB>0, np.sqrt(pAB), 0.)

def blatt_weisskopf_np(pR, pAB,radius,spin):
    r2 = radius*radius
    pAB2 = pAB*pAB
    pR2 = pR*pR
    bf= np.where( spin == 1, np.sqrt(1.0+r2*pR*pR)/np.sqrt(1.0+r2*pAB2),
                  np.where( spin == 2, np.sqrt(9.0+3.0*r2*pR2+r2*pR2*r2*pR2)/np.sqrt(9.0+3.0*r2*pAB2+r2*pAB2*r2*pAB2), 
                            1.0) )
    return bf

def amplitude_BW(v1, v2, v3,
                 ampl, phase, gamma, mass, spin,
                 mA, mB, mC, mMother):

    R_r = 1.5 # "resonance radius" for Blatt-Weisskoper barrier factors.
    R_D = 5.0 # "D meson radius" for Blatt-Weisskoper barrier factors.

    mAB=np.sqrt(v1) # = (_p4_d1+_p4_d2).mass();
    mBC=np.sqrt(v2) # = (_p4_d2+p4_d3).mass();
    mAC=np.sqrt(v3) # = (_p4_d1+p4_d3).mass();

    pAB=two_body_momentum_np(mAB, mA, mB)
    pR =two_body_momentum_np(mass, mA, mB)

    pMother=two_body_momentum_np(mMother, mass, mC)
    pDAB=two_body_momentum_np(mMother, mAB, mC)

    # Compute Blatt-Weisskopf barrier factors.
    fR= blatt_weisskopf_np(pR, pAB, R_r, spin)
    fD= blatt_weisskopf_np(pMother, pDAB, R_D, spin)
    power= 2.0*spin+1.0
    
    # Compute the running width.
    gammaAB= gamma*np.power(pAB/pR,power)*(mass/mAB)*fR*fR

    # Compute the amplitude.
    matrixEl = helicity_np(mAB,mBC,mAC,mA,mB,mC,mMother,spin)*fR*fD/(mass*mass-mAB*mAB-mass*gammaAB*1j)
    matrixEl *= ampl*complex(np.cos(phase*pi180inv),np.sin(phase*pi180inv))

    return matrixEl

Plot BW

In [ ]:
nnorm=100000
norm_sample = phsp.uniform_sample(nnorm)
m2ac = phsp.m2ac(norm_sample)
m2ab = phsp.m2ab(norm_sample)
m2bc = phsp.m2bc(norm_sample)
m_ac = np.sqrt(m2ac)
m_ab = np.sqrt(m2ab)
m_bc = np.sqrt(m2bc)

Calculate BW amplitude for $\rho(770)$

In [ ]:
mrho = atfi.const(lp.rho_770_0.mass/1000)
wrho = atfi.const(lp.rho_770_0.width/1000)
srho = atfi.const(lp.rho_770_0.S)
amp_rho = atfi.const(1.0)
ph_rho = atfi.const(0.0)
bw_rho_amp = amplitude_BW(m2ac, m2ab, m2bc,
                 amp_rho.numpy(), ph_rho.numpy(), wrho.numpy(), mrho.numpy(), srho.numpy(),
                 mpi.numpy(), mpi.numpy(), mkz.numpy(), md.numpy())

In [ ]:
plt.hist(m_ac, weights = density_np(bw_rho_amp), bins=100, histtype='step', label=r'$\rho(770)$ Jordi')
#plt.hist(phsp.m2ab(norm_sample), weights = atfd.density(aBW_tf), bins=100, histtype='step', label='BW Amplitf', density=True)
plt.xlabel(r'$m_{ac}^2$ [GeV$^2$]')
plt.ylabel(r'$|A|$')
plt.title('Amplitude vs m2ac')
plt.legend()

AmpliTF Implementation

- test difference in `two_body_momentum`

In [ ]:
test = two_body_momentum_np(m_ac, mpi, mpi) - atfk.two_body_momentum(m_ac, mpi, mpi)
print('Difference in two_body_momentum_np and two_body_momentum:', atfi.sum(test))

- test difference in `blatt_weisskopf_ff`

In [ ]:
test = blatt_weisskopf_np(two_body_momentum_np(mrho, mpi, mpi), two_body_momentum_np(m_ac, mpi, mpi), 1.5, srho) - \
    atfd.blatt_weisskopf_ff(atfk.two_body_momentum(m_ac, mpi, mpi), atfk.two_body_momentum(mrho, mpi, mpi), atfi.const(1.5), 1)
print('Difference in blatt_weisskopf form factors:', atfi.sum(test))

- test difference in helicity distribution

In [ ]:
hel_np = helicity_np(m2ac, m2ab, m2bc,
                     mpi.numpy(), mpi.numpy(), mkz.numpy(), md.numpy(), 1)
hel_tf = atfd.helicity_amplitude(phsp.cos_helicity_ac(norm_sample), 1)
plt.hist(m2ac, weights=density_np(hel_np), bins=100, histtype='step', label='Helicity Amplitude NumPy')
plt.hist(m2ac, weights=atfd.density(hel_tf), bins=100, histtype='step', label='Helicity Amplitude Amplitf')
plt.legend()

Jordi's helicity distribution is defined as:
- spin-0: $1$
- spin-1: $m_{AC}^2-m_{BC}^2+\frac{(m_{Mother}^2-m_C^2)(m_B^2-m_A^2)}{m_{AB}^2}$
- spin-2: $\left(m_{BC}^2-m_{AC}^2+\frac{(m_{Mother}^2-m_C^2)(m_A^2-m_B^2)}{m_{AB}^2}\right)^2-
                                    \frac{m_{AB}^2-2(m_{Mother}^2+m_C^2)+(m_{Mother}^2-m_C^2)^2}{3m_{AB}^2}
                                    \frac{m_{AB}^2-2(m_A^2+m_B^2)+(m_A^2-m_B^2)^2}{m_{AB}^2}$


In [ ]:
def helicity_tf(mAB,mBC,mAC,mA,mB,mC,mD,spin):
    hel = atfi.where( spin==1, mAC*mAC-mBC*mBC+((mD*mD-mC*mC)*(mB*mB-mA*mA)/(mAB*mAB)),
                  atfi.where( spin==2, atfi.pow(mBC*mBC-mAC*mAC+(mD*mD-mC*mC)*(mA*mA-mB*mB)/(mAB*mAB),2)-
                                    1./3.*(mAB*mAB-2.*(mD*mD+mC*mC)+atfi.pow(mD*mD-mC*mC,2)/(mAB*mAB))*
                                    (mAB*mAB-2.*(mA*mA+mB*mB)+atfi.pow(mA*mA-mB*mB,2)/(mAB*mAB)),
                             1.0) )    
    return hel

Test differences in relativistic BW

In [ ]:
def rel_BW_jd_tf(v1, v2, v3,
                 ampl, phase, gamma, mass, spin,
                 mA, mB, mC, mMother, mAAlt=None, mMotherAlt=None):

    R_r = 1.5 # "resonance radius" for Blatt-Weisskoper barrier factors.
    R_D = 5.0 # "D meson radius" for Blatt-Weisskoper barrier factors.

    mAB=atfi.sqrt(v1) # = (_p4_d1+_p4_d2).mass();
    mBC=atfi.sqrt(v2) # = (_p4_d2+p4_d3).mass();
    mAC=atfi.sqrt(v3) # = (_p4_d1+p4_d3).mass();

    p =atfk.two_body_momentum(mAB, mA, mB)
    p0 =atfk.two_body_momentum(mass, mA, mB)

    pDAB=atfk.two_body_momentum(mMother, mAB, mC)
    pMother=atfk.two_body_momentum(mMother if mMotherAlt is None else mMotherAlt, mass, mC)

    # Compute Blatt-Weisskopf barrier factors.
    ffr= atfd.blatt_weisskopf_ff(p, p0, R_r, spin)
    fD= atfd.blatt_weisskopf_ff(pDAB, pMother, R_D, spin)
    
    # Compute the running width.
    gammaAB = atfd.mass_dependent_width(mAB, mass, gamma, p, p0, ffr, spin)

    # Compute the amplitude.
    #matrixEl = atfi.cast_complex(ffr*fD)*atfd.relativistic_breit_wigner(v1,mass,gammaAB)
    #matrixEl *= atfi.cast_complex(helicity_tf(mAB,mBC,mAC,mA,mB,mC,mMother,spin))
    matrixEl = atfi.cast_complex(helicity_tf(mAB,mBC,mAC,mA,mB,mC,mMother,spin)*ffr*fD)/atfi.complex(mass*mass-mAB*mAB,-mass*gammaAB)
    #    matrixEl = atfi.cast_complex(ffr*fD)*atfd.relativistic_breit_wigner(v1,mass,gammaAB)
    #matrixEl = atfd.relativistic_breit_wigner(v1,mass,gammaAB)

    return matrixEl

@atfi.function
def breit_wigner_lineshape(
    m2,
    m0,
    gamma0,
    ma,
    mb,
    mc,
    md,
    dr,
    dd,
    lr,
    ld,
    barrier_factor=True,
    ma0=None,
    md0=None,
):
    m = atfi.sqrt(m2)
    q = atfk.two_body_momentum(md, m, mc)
    q0 = atfk.two_body_momentum(md if md0 is None else md0, m0, mc)
    p = atfk.two_body_momentum(m, ma, mb)
    p0 = atfk.two_body_momentum(m0, ma if ma0 is None else ma0, mb)
    ffr = atfd.blatt_weisskopf_ff(p, p0, dr, lr)
    ffd = atfd.blatt_weisskopf_ff(q, q0, dd, ld)
    width = atfd.mass_dependent_width(m, m0, gamma0, p, p0, ffr, lr)
    bw = atfd.relativistic_breit_wigner(m2, m0, width)  
    ff = ffr * ffd
    if barrier_factor:
        b1 = atfd.orbital_barrier_factor(p, p0, lr)
        b2 = atfd.orbital_barrier_factor(q, q0, ld)
        ff *= b1 * b2
    return bw * atfi.complex(ff, atfi.const(0.0))

In [ ]:
bw_rho_shape_tf = breit_wigner_lineshape(m2ac, mrho, wrho, 
                                             mpi, mpi, mkz, md,
                                             1.5, 5.0, 1, 1, barrier_factor=False) * atfi.cast_complex(helicity_tf(m_ac,m_bc,m_ab,mpi,mpi,mkz,md,1))
bw_rho_shape = rel_BW_jd_tf(m2ac, m2ab, m2bc,
                 amp_rho, ph_rho, wrho, mrho, 1,
                 mpi, mpi, mkz, md)#* atfi.cast_complex(helicity_tf(m_ac,m_bc,m_ab,mpi,mpi,mkz,md,1))

In [ ]:
plt.hist(m_ac, weights = density_np(bw_rho_shape), bins=100, histtype='step', label=r'$\rho(770)$ Shape Jordi')
plt.hist(m_ac, weights = atfd.density(bw_rho_shape_tf), bins=100, histtype='step', label=r'$\rho(770)$ Shape AmpliTF')
#plt.hist(m_ac, weights = atfd.density(bw_rho_amp_tf2), bins=100, histtype='step', label=r'$\rho(770)$ AmpliTF2')
#plt.hist(phsp.m2ab(norm_sample), weights = atfd.density(aBW_tf), bins=100, histtype='step', label='BW Amplitf', density=True)
plt.xlabel(r'$m_{ac}$ [GeV]')
plt.ylabel(r'$|A|$')
plt.title('Amplitude vs m_ac')
plt.legend()

Let's code the BW the way Jordi did

In [ ]:



def amplitude_BW_tf(v1, v2, v3,
                 ampl, phase, gamma, mass, spin,
                 mA, mB, mC, mMother, mAAlt=None, mMotherAlt=None):

    R_r = 1.5 # "resonance radius" for Blatt-Weisskoper barrier factors.
    R_D = 5.0 # "D meson radius" for Blatt-Weisskoper barrier factors.

    mAB=atfi.sqrt(v1) # = (_p4_d1+_p4_d2).mass();
    mBC=atfi.sqrt(v2) # = (_p4_d2+p4_d3).mass();
    mAC=atfi.sqrt(v3) # = (_p4_d1+p4_d3).mass();

    pAB=atfk.two_body_momentum(mAB, mA, mB)
    pR =atfk.two_body_momentum(mass, mA, mB)

    pMother=atfk.two_body_momentum(mMother if mMotherAlt is None else mMotherAlt, mass, mC)
    pDAB=atfk.two_body_momentum(mMother, mAB, mC)

    # Compute Blatt-Weisskopf barrier factors.
    fR= atfd.blatt_weisskopf_ff(pAB, pR, R_r, spin)
    fD= atfd.blatt_weisskopf_ff(pDAB, pMother, R_D, spin)
    
    # Compute the running width.
    gammaAB = atfd.mass_dependent_width(mAB, mass, gamma, pAB, pR, fR, spin)

    # Compute the amplitude.
    #matrixEl = atfi.cast_complex(helicity_tf(mAB,mBC,mAC,mA,mB,mC,mMother,spin)*fR*fD)/atfi.complex(mass*mass-mAB*mAB,-mass*gammaAB)
    matrixEl = atfi.cast_complex(helicity_tf(mAB,mBC,mAC,mA,mB,mC,mMother,spin)*fR*fD)*atfd.relativistic_breit_wigner(v1,mass,gammaAB)
    matrixEl *= atfi.complex(ampl*atfi.cos(phase*pi180inv),ampl*atfi.sin(phase*pi180inv))

    return matrixEl

In [ ]:
bw_rho_amp_tf = amplitude_BW_tf(m2ac, m2bc, m2ab,
                 amp_rho, ph_rho, wrho, mrho, 1,
                 mpi, mpi, mkz, md)

In [ ]:
bw_rho_amp_tf2 = atfd.breit_wigner_lineshape(m2ac, mrho, wrho, 
                                             mpi, mpi, mkz, md,
                                             1.5, 5.0, 1, 1, barrier_factor=False) * \
                atfi.cast_complex(helicity_tf(m_ac,m_bc,m_ab,mpi,mpi,mkz,md,1)) * \
                atfi.complex(amp_rho*atfi.cos(ph_rho*pi180inv),amp_rho*atfi.sin(ph_rho*pi180inv))

In [ ]:
plt.hist(m_ac, weights = density_np(bw_rho_amp), bins=100, histtype='step', label=r'$\rho(770)$ Jordi')
plt.hist(m_ac, weights = atfd.density(bw_rho_amp_tf), bins=100, histtype='step', label=r'$\rho(770)$ AmpliTF')
plt.hist(m_ac, weights = atfd.density(bw_rho_amp_tf2), bins=100, histtype='step', label=r'$\rho(770)$ AmpliTF2')
#plt.hist(phsp.m2ab(norm_sample), weights = atfd.density(aBW_tf), bins=100, histtype='step', label='BW Amplitf', density=True)
plt.xlabel(r'$m_{ac}$ [GeV]')
plt.ylabel(r'$|A|$')
plt.title('Amplitude vs m_ac')
plt.legend()

## Comparison with the C++ code

The C++ implementation (`Resonance.cpp` in d02kspipi_toys) is no longer loaded with ROOT here. Its values at 2000 Dalitz points are stored in `inputs/d02kspipi_cpp_reference.npz` (produced by `macros/make_cpp_reference.py`), and `tests/test_cpp_reference.py` checks every term of the model against them. The comparison uses the Belle 2018 parameters and the masses of the C++ code. The previous ROOT-based cells are in the git history of `main`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tfaex.reference import load_cpp_reference, tf_components
from tfaex.plotting import plot_reference_comparison

ref = load_cpp_reference()
tf = tf_components(ref)
m_pipi, m_kspim, m_kspip = (np.sqrt(ref[k]) for k in ("m2_pipi", "m2_kspim", "m2_kspip"))

Breit-Wigner terms in $\pi\pi$: $\rho(770)$, $\omega(782)$, $\rho(1450)$

In [ ]:
for name in ["rho770", "omega", "rho1450"]:
    plot_reference_comparison(m_pipi, ref[name], tf[name], r"$m(\pi\pi)$ [GeV]", title=name)

$K^*(892)$: Cabibbo-favoured in $K_S^0\pi^-$, doubly Cabibbo-suppressed in $K_S^0\pi^+$

In [ ]:
plot_reference_comparison(m_kspim, ref["Kstar892minus"], tf["Kstar892minus"], r"$m(K_S^0\pi^-)$ [GeV]", title="Kstar892minus")
plot_reference_comparison(m_kspip, ref["Kstar892plus"], tf["Kstar892plus"], r"$m(K_S^0\pi^+)$ [GeV]", title="Kstar892plus");

Full amplitude. The remaining differences come from the K-matrix below $m^2(\pi\pi) = 1$ GeV$^2$ (see the K-matrix notebook).

In [ ]:
fig, ax = plot_reference_comparison(m_pipi, ref["total"], tf["total"], r"$m(\pi\pi)$ [GeV]", title="total")
ax[2].axvline(1.0, color="gray", ls="--");